# 계산엔진 정리 및 기본 위험등급

이 노트북을 위에서 아래로 실행합니다. CSV를 읽기만 하므로 `모두 실행`해도 데이터를 덮어쓰지 않습니다.

공통 처리(질문 조회, 숫자 검증, 규칙 매칭, 결과 생성)를 한 번씩 정의합니다. 이전 노트북과 다른 커널 상태에 의존하지 않습니다.

현재 결과는 **적용조건이 충족됐다고 가정한 시험 계산**입니다. 기본등급은 잠정값이며, 법령 경고·조건부 질문·MSDS 파생변수는 아직 구현 전입니다. 이전 노트북의 내용은 그대로 보존했습니다.


In [ ]:
from pathlib import Path
from collections.abc import Mapping
import math
import pandas as pd
from pandas.api.types import is_scalar, is_bool

# 실행 위치가 프로젝트 루트 또는 notebooks 폴더여도 data 폴더를 찾습니다.
def find_data_dir():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for candidate in [parent / "data", parent / "fire_risk_project" / "data"]:
            if (candidate / "08_점수규칙DB.csv").is_file():
                return candidate
    raise FileNotFoundError("프로젝트 data 폴더를 찾을 수 없습니다.")

data_dir = find_data_dir()

def read_database(filename):
    table = pd.read_csv(
        data_dir / filename, encoding="utf-8-sig",
        dtype=str, keep_default_na=False,
    )
    # ID와 선택지 양끝의 불필요한 공백만 정리합니다.
    return table.apply(lambda column: column.str.strip())

tasks = read_database("01_공정_세부작업DB.csv")
questions = read_database("02_위험요인_입력항목DB.csv")
score_rules = read_database("08_점수규칙DB.csv")
standards = read_database("04_법령_KOSHA기준DB.csv")

print("읽은 폴더:", data_dir)
print(f"작업 {len(tasks)}개 / 질문 {len(questions)}개 / 규칙 조건 {len(score_rules)}개")


## 1. 공통 함수

- `get_question()`: 질문 조회를 한곳에서 처리합니다.
- `to_number()`: 빈칸·무한대·참/거짓 등을 숫자로 잘못 계산하지 않도록 합니다.
- `matches_numeric_rule()`: CSV에 저장된 비교연산자와 구간 경계를 해석합니다.
- `build_score_result()`: 선택형과 숫자형이 같은 결과 형식을 사용하게 합니다.

점수·가중치·정규화 기준·등급 경계는 CSV에서 읽습니다. CSV의 규칙 문장을 임의로 실행하지 않습니다.


In [2]:
def split_ids(value):
    return [item.strip() for item in value.split(";") if item.strip()]

def is_missing(value):
    if not is_scalar(value):
        raise ValueError("응답은 목록이나 사전이 아닌 단일 값이어야 합니다.")
    return bool(pd.isna(value)) or (isinstance(value, str) and not value.strip())

def to_number(value, label):
    if is_bool(value) or is_missing(value):
        raise ValueError(f"{label}: 숫자가 필요합니다.")
    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        raise ValueError(f"{label}: 숫자로 변환할 수 없습니다.") from None
    if not math.isfinite(number):
        raise ValueError(f"{label}: 유한한 숫자가 필요합니다.")
    return number

def read_flag(value, label):
    number = to_number(value, label)
    if number not in (0, 1):
        raise ValueError(f"{label}: 0 또는 1이어야 합니다.")
    return number == 1

def get_single_row(table, label):
    if len(table) != 1:
        raise ValueError(f"{label}: 해당 행이 없거나 중복됩니다.")
    return table.iloc[0]

def get_question(question_id):
    question_id = str(question_id).strip()
    return get_single_row(
        questions.loc[questions["항목ID"] == question_id],
        f"질문 {question_id}",
    )

def get_rules(rule_id):
    result = score_rules.loc[score_rules["점수규칙ID"] == rule_id]
    if result.empty:
        raise ValueError(f"존재하지 않는 점수규칙: {rule_id}")
    return result

def matches_numeric_rule(number, rule):
    operator = rule["비교연산자"]
    lower = to_number(rule["기준값1"], "기준값1")

    if operator == "<":
        return number < lower
    if operator == "<=":
        return number <= lower
    if operator == ">":
        return number > lower
    if operator == ">=":
        return number >= lower
    if operator == "EQ":
        return number == lower
    if operator == "구간":
        upper = to_number(rule["기준값2"], "기준값2")
        if lower > upper:
            raise ValueError("구간의 하한이 상한보다 큽니다.")
        lower_included = read_flag(rule["하한포함"], "하한포함")
        upper_included = read_flag(rule["상한포함"], "상한포함")
        lower_ok = number >= lower if lower_included else number > lower
        upper_ok = number <= upper if upper_included else number < upper
        return lower_ok and upper_ok
    raise NotImplementedError(f"지원하지 않는 비교연산자: {operator}")

def build_score_result(question, answer, rule):
    score = to_number(rule["점수"], "점수")
    weight = to_number(rule["가중치"], "가중치")
    maximum = to_number(rule["정규화최대점수"], "정규화최대점수")
    if maximum <= 0 or weight < 0 or not 0 <= score <= maximum:
        raise ValueError("점수·가중치·정규화 기준의 범위를 확인하세요.")

    included = weight > 0
    evidence = list(dict.fromkeys(
        split_ids(question["연결기준ID"]) + split_ids(rule["근거ID"])
    ))
    return {
        "항목ID": question["항목ID"],
        "확인항목": question["확인항목"],
        "사용자응답": answer,
        "단위": question["단위"],
        "점수규칙ID": rule["점수규칙ID"],
        "조건순번": rule["조건순번"],
        "점수": score,
        "가중치": weight,
        "가중점수": score * weight if included else 0.0,
        "최대가중점수": maximum * weight if included else 0.0,
        "점수합산대상": included,
        "중복집계그룹": question["중복집계그룹"],
        "그룹집계방식": question["그룹집계방식"],
        "관련기준ID": ";".join(evidence),
        "적용조건": rule["적용조건"],
        "확정상태": rule["확정상태"],
    }


## 2. DB 연결 검사와 작업별 질문

잘못된 ID·중복 규칙·필수여부를 먼저 검사합니다. H05는 질문 9개가 나와야 합니다.


In [3]:
def validate_databases():
    requirements = [
        (tasks, ["작업ID"]),
        (questions, ["항목ID", "적용작업ID", "확인항목", "입력형식",
                     "선택지·허용값", "단위", "필수여부", "점수규칙ID",
                     "연결기준ID", "중복집계그룹", "그룹집계방식"]),
        (score_rules, ["점수규칙ID", "조건순번", "규칙유형", "입력값·조건",
                       "비교연산자", "기준값1", "기준값2", "점수", "가중치",
                       "정규화최대점수", "하한포함", "상한포함",
                       "근거ID", "적용조건", "확정상태"]),
        (standards, ["기준ID"]),
    ]
    for table, columns in requirements:
        missing = set(columns) - set(table.columns)
        if missing:
            raise ValueError(f"필수 컬럼 누락: {sorted(missing)}")
    for table, key in [(tasks, "작업ID"), (questions, "항목ID"), (standards, "기준ID")]:
        if table[key].eq("").any() or table[key].duplicated().any():
            raise ValueError(f"{key}: 빈값 또는 중복")

    sequences = score_rules["조건순번"].map(lambda x: to_number(x, "조건순번"))
    if not sequences.map(lambda x: x >= 1 and x.is_integer()).all():
        raise ValueError("조건순번은 1 이상의 정수여야 합니다.")
    rule_keys = list(zip(score_rules["점수규칙ID"], sequences))
    if score_rules["점수규칙ID"].eq("").any() or len(set(rule_keys)) != len(rule_keys):
        raise ValueError("규칙ID가 비었거나 규칙ID+조건순번이 중복됩니다.")

    task_ids = set(tasks["작업ID"])
    rule_ids = set(score_rules["점수규칙ID"])
    standard_ids = set(standards["기준ID"])
    for _, question in questions.iterrows():
        if not split_ids(question["적용작업ID"]):
            raise ValueError(f"{question['항목ID']}: 적용작업ID 누락")
        if set(split_ids(question["적용작업ID"])) - task_ids:
            raise ValueError(f"{question['항목ID']}: 연결된 작업ID 오류")
        if question["점수규칙ID"] not in rule_ids:
            raise ValueError(f"{question['항목ID']}: 연결된 점수규칙ID 오류")
        if set(split_ids(question["연결기준ID"])) - standard_ids:
            raise ValueError(f"{question['항목ID']}: 연결된 기준ID 오류")
        read_flag(question["필수여부"], "필수여부")
        if not question["중복집계그룹"] or question["그룹집계방식"] != "MAX_PER_TASK":
            raise ValueError(f"{question['항목ID']}: 중복집계 설정 오류")

    for _, rule in score_rules.iterrows():
        if set(split_ids(rule["근거ID"])) - standard_ids:
            raise ValueError(f"{rule['점수규칙ID']}: 근거ID 오류")
    print("DB의 기본 연결·중복·집계 설정 검사 통과")

def get_questions_for_task(task_id):
    task_id = str(task_id).strip()
    get_single_row(tasks.loc[tasks["작업ID"] == task_id], f"작업 {task_id}")
    applies = questions["적용작업ID"].map(lambda value: task_id in split_ids(value))
    return questions.loc[applies].copy().reset_index(drop=True)

validate_databases()
h05_questions = get_questions_for_task("H05")
print("H05 질문 수:", len(h05_questions))
display(h05_questions[["항목ID", "확인항목", "입력형식", "점수규칙ID"]])


DB의 기본 연결·중복·집계 설정 검사 통과
H05 질문 수: 9


,항목ID,확인항목,입력형식,점수규칙ID
0,Q001,주변 가연물 존재 여부,선택,S001
1,Q002,가연물과 작업점 거리,숫자,S002
2,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,선택,S003
3,Q004,화재감시자 배치 여부,선택,S004
4,Q018,용접방화포 사용 시 성능인증 제품 여부,선택,S019
5,Q019,작업점 하부 가연물의 불티 착화 우려 여부,선택,S020
6,Q020,금속 칸막이·벽·천장·지붕 반대면 가연물의 열전도·복사 착화 우려 여부,선택,S021
7,Q025,초기소화설비 비치·접근 가능 여부,선택,S026
8,Q026,작업 종료 후 잔불·불티 확인 여부,선택,S027


## 3. 응답 점수화

이제 `score_answer()` 하나가 입력형식을 확인해 선택형 또는 숫자형 규칙을 찾습니다. 앞서 만든 별도의 `score_numeric_answer()`를 호출할 필요가 없습니다.

가스농도 Q030처럼 원시 입력을 파생변수로 변환해야 하는 질문은 아직 처리하지 않으며, 명시적인 오류로 알려줍니다.


In [4]:
def score_answer(question_id, answer):
    question = get_question(question_id)
    if is_missing(answer):
        raise ValueError(f"{question['항목ID']}: 응답이 비었습니다.")
    rules = get_rules(question["점수규칙ID"])

    if question["입력형식"] == "선택":
        answer = str(answer).strip()
        allowed = [value.strip() for value in question["선택지·허용값"].split("/")]
        if answer not in allowed:
            raise ValueError(f"{question['항목ID']} 허용 응답: {allowed}")
        if not ((rules["규칙유형"] == "범주형") & (rules["비교연산자"] == "EQ")).all():
            raise ValueError("선택형 질문의 규칙유형·연산자를 확인하세요.")
        matched = rules.loc[rules["입력값·조건"] == answer]

    elif question["입력형식"] == "숫자":
        if "/" in question["단위"] or not rules["적용대상"].eq("02_위험요인").all():
            raise NotImplementedError("단위 환산·MSDS 파생변수 계산이 먼저 필요합니다.")
        answer = to_number(answer, question["확인항목"])
        if question["선택지·허용값"] != "0 이상":
            raise NotImplementedError("지원하지 않는 숫자 입력 제한입니다.")
        if answer < 0:
            raise ValueError("0 이상의 숫자를 입력하세요.")
        if not rules["규칙유형"].isin(["수치형", "범위형"]).all():
            raise ValueError("숫자형 질문의 규칙유형을 확인하세요.")
        mask = rules.apply(lambda rule: matches_numeric_rule(answer, rule), axis=1)
        matched = rules.loc[mask]
    else:
        raise NotImplementedError(f"지원하지 않는 입력형식: {question['입력형식']}")

    rule = get_single_row(matched, f"{question['점수규칙ID']} 응답 {answer}")
    return build_score_result(question, answer, rule)

display(pd.DataFrame([
    score_answer("Q004", "미배치"),
    score_answer("Q004", "해당없음"),
    score_answer("Q002", 10.9),
    score_answer("Q002", 11),
]))


,항목ID,확인항목,사용자응답,단위,점수규칙ID,조건순번,점수,가중치,가중점수,최대가중점수,점수합산대상,중복집계그룹,그룹집계방식,관련기준ID,적용조건,확정상태
0,Q004,화재감시자 배치 여부,미배치,-,S004,1,3.0,2.0,6.0,6.0,True,Q004,MAX_PER_TASK,R002,법령상 배치대상 조건 충족 시 적용,미확정
1,Q004,화재감시자 배치 여부,해당없음,-,S004,3,0.0,0.0,0.0,0.0,False,Q004,MAX_PER_TASK,R002,법령상 배치대상 조건 충족 시 적용,미확정
2,Q002,가연물과 작업점 거리,10.9,m,S002,1,1.0,1.5,1.5,4.5,True,Q002,MAX_PER_TASK,R001;R002,화기·불티 작업,미확정
3,Q002,가연물과 작업점 거리,11.0,m,S002,2,0.0,1.5,0.0,4.5,True,Q002,MAX_PER_TASK,R001;R002,화기·불티 작업,미확정


## 4. 필수응답, 중복 집계, 기본등급

- 공백이 붙은 질문ID는 정리하되, 정리 후 같은 ID가 두 번 나오면 거부합니다.
- 필수질문은 CSV 기준으로 검사합니다. 조건부 질문 제외는 후속 단계입니다.
- 동일 작업·집계그룹에서는 가중점수가 가장 높은 응답 한 건만 사용합니다.
- 등급은 S045의 경계와 포함 여부를 사용합니다. 법령 경고를 검사하지 않았으므로 최종판정과 분리합니다.


In [5]:
def normalize_answers(answers):
    if not isinstance(answers, Mapping):
        raise ValueError("응답은 {질문ID: 응답} 형식의 사전이어야 합니다.")
    normalized = {}
    for key, value in answers.items():
        key = str(key).strip()
        if not key or key in normalized:
            raise ValueError(f"빈 질문ID 또는 공백 정리 후 중복된 질문ID: {key}")
        if not is_scalar(value):
            raise ValueError(f"{key}: 단일 응답값이 필요합니다.")
        normalized[key] = value
    return normalized

def check_required_answers(task_id, answers):
    answers = normalize_answers(answers)
    task_questions = get_questions_for_task(task_id)
    required = task_questions["필수여부"].map(lambda value: read_flag(value, "필수여부"))
    missing_ids = [
        question_id
        for question_id in task_questions.loc[required, "항목ID"]
        if is_missing(answers.get(question_id))
    ]
    missing = task_questions.loc[
        task_questions["항목ID"].isin(missing_ids), ["항목ID", "확인항목"]
    ].reset_index(drop=True)
    unexpected = sorted(set(answers) - set(task_questions["항목ID"]))
    summary = {
        "작업ID": str(task_id).strip(),
        "필수질문수": int(required.sum()),
        "누락질문수": len(missing),
        "작업외질문ID": unexpected,
        "필수응답입력완료": missing.empty and not unexpected,
    }
    return summary, missing

def select_representatives(detail):
    candidates = detail.loc[detail["점수합산대상"]]
    return (
        candidates.sort_values(["가중점수", "항목ID"], ascending=[False, True])
        .drop_duplicates("중복집계그룹", keep="first")
    )

def get_basic_grade(risk_index):
    if risk_index is None:
        return "평가불가"
    number = to_number(risk_index, "위험지수")
    rules = get_rules("S045")
    matched = rules.loc[
        rules.apply(lambda rule: matches_numeric_rule(number, rule), axis=1)
    ]
    rule = get_single_row(matched, f"위험지수 {number}의 기본등급")
    return rule["입력값·조건"]

def calculate_task_risk(task_id, answers):
    answers = normalize_answers(answers)
    check, missing = check_required_answers(task_id, answers)
    if check["작업외질문ID"]:
        raise ValueError(f"작업 외 질문: {check['작업외질문ID']}")
    if not missing.empty:
        raise ValueError(f"필수응답 누락: {missing['항목ID'].tolist()}")

    # 선택사항을 비워둔 것은 제외하지만 실제 숫자 0은 보존합니다.
    results = [
        score_answer(question_id, answer)
        for question_id, answer in answers.items()
        if not is_missing(answer)
    ]
    if not results:
        raise ValueError("점수화할 응답이 없습니다.")
    detail = pd.DataFrame(results)
    representatives = select_representatives(detail)
    detail["집계대표여부"] = detail.index.isin(representatives.index)

    total = float(representatives["가중점수"].sum())
    maximum = float(representatives["최대가중점수"].sum())
    risk_index = total / maximum * 100 if maximum > 0 else None
    summary = {
        "작업ID": str(task_id).strip(),
        "입력질문수": len(detail),
        "집계항목수": len(representatives),
        "총가중위험점수": total,
        "최대가중점수": maximum,
        "위험지수": risk_index,
        "기본위험등급(잠정)": get_basic_grade(risk_index),
        "강제경고": "미검증",
        "최종판정": "보류: 적용조건·법령 미검증",
        "계산상태": "시험 계산" if maximum > 0 else "계산 불가: 합산 대상 없음",
    }
    return summary, detail

def get_top_risks(detail, count=3):
    if isinstance(count, bool) or not isinstance(count, int) or count < 1:
        raise ValueError("표시 개수는 1 이상의 정수여야 합니다.")
    positive = detail.loc[detail["집계대표여부"] & (detail["가중점수"] > 0)]
    return (
        positive.sort_values(["가중점수", "항목ID"], ascending=[False, True])
        .head(count)[["항목ID", "확인항목", "사용자응답", "가중점수", "관련기준ID"]]
        .reset_index(drop=True)
    )


## 5. H05 전체 시험 계산

아래는 이전 대화와 동일한 **가상 응답**입니다. 위험지수 약 40.91, 기본등급 '주의'가 나와야 합니다. 기본등급이 '주의'여도 법령상 조치 필요 여부는 별도이며, 최종판정은 보류합니다.


In [6]:
h05_answers = {
    "Q001": "있음",
    "Q002": 5,
    "Q003": "미실시",
    "Q004": "미배치",
    "Q018": "용접방화포 미사용",
    "Q019": "없음",
    "Q020": "없음",
    "Q025": "적정",
    "Q026": "완료",
}
summary, detail = calculate_task_risk("H05", h05_answers)
display(pd.DataFrame([summary]))
display(detail)
display(get_top_risks(detail))


,작업ID,입력질문수,집계항목수,총가중위험점수,최대가중점수,위험지수,기본위험등급(잠정),강제경고,최종판정,계산상태
0,H05,9,8,13.5,33.0,40.909091,주의,미검증,보류: 적용조건·법령 미검증,시험 계산


,항목ID,확인항목,사용자응답,단위,점수규칙ID,조건순번,점수,가중치,가중점수,최대가중점수,점수합산대상,중복집계그룹,그룹집계방식,관련기준ID,적용조건,확정상태,집계대표여부
0,Q001,주변 가연물 존재 여부,있음,-,S001,1,1.0,1.5,1.5,4.5,True,Q001,MAX_PER_TASK,R001,화기·불티/전기/열원 작업 공통,미확정,True
1,Q002,가연물과 작업점 거리,5.0,m,S002,1,1.0,1.5,1.5,4.5,True,Q002,MAX_PER_TASK,R001;R002,화기·불티 작업,미확정,True
2,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,미실시,-,S003,1,3.0,1.5,4.5,4.5,True,Q003,MAX_PER_TASK,R001,화기·불티 작업,미확정,True
3,Q004,화재감시자 배치 여부,미배치,-,S004,1,3.0,2.0,6.0,6.0,True,Q004,MAX_PER_TASK,R002,법령상 배치대상 조건 충족 시 적용,미확정,True
4,Q018,용접방화포 사용 시 성능인증 제품 여부,용접방화포 미사용,-,S019,3,0.0,0.0,0.0,0.0,False,Q018,MAX_PER_TASK,R001,다른 적정 비산방지조치를 적용하는 경우,미확정,False
5,Q019,작업점 하부 가연물의 불티 착화 우려 여부,없음,-,S020,2,0.0,1.0,0.0,3.0,True,Q019,MAX_PER_TASK,R002,다른 배치조건도 함께 확인,미확정,True
6,Q020,금속 칸막이·벽·천장·지붕 반대면 가연물의 열전도·복사 착화 우려 여부,없음,-,S021,2,0.0,1.0,0.0,3.0,True,Q020,MAX_PER_TASK,R002,다른 배치조건도 함께 확인,미확정,True
7,Q025,초기소화설비 비치·접근 가능 여부,적정,-,S026,1,0.0,1.5,0.0,4.5,True,Q025,MAX_PER_TASK,R001;R014,화재위험 작업,미확정,True
8,Q026,작업 종료 후 잔불·불티 확인 여부,완료,-,S027,1,0.0,1.0,0.0,3.0,True,Q026,MAX_PER_TASK,R015,용접·용단·그라인딩 등 화기·불티 작업,미확정,True


,항목ID,확인항목,사용자응답,가중점수,관련기준ID
0,Q004,화재감시자 배치 여부,미배치,6.0,R002
1,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,미실시,4.5,R001
2,Q001,주변 가연물 존재 여부,있음,1.5,R001


## 6. 경계값과 누락 확인

S045의 25·50·75·100 경계와 거리 11m를 확인합니다. 아래 숫자는 현재 DB의 **검증 예시**이며, 등급 계산 함수의 판단 기준은 CSV에서 읽습니다.


In [7]:
boundary_results = [
    {"위험지수": value, "기본등급": get_basic_grade(value)}
    for value in [0, 24.99, 25, 49.99, 50, 74.99, 75, 100]
]
display(pd.DataFrame(boundary_results))

partial = {"Q003": "미실시", "Q004": "배치"}
missing_summary, missing_questions = check_required_answers("H05", partial)
display(pd.DataFrame([missing_summary]))
display(missing_questions)


,위험지수,기본등급
0,0.00,낮음
1,24.99,낮음
2,25.00,주의
3,49.99,주의
4,50.00,높음
5,74.99,높음
6,75.00,매우높음
7,100.00,매우높음


,작업ID,필수질문수,누락질문수,작업외질문ID,필수응답입력완료
0,H05,9,7,[],False


,항목ID,확인항목
0,Q001,주변 가연물 존재 여부
1,Q002,가연물과 작업점 거리
2,Q018,용접방화포 사용 시 성능인증 제품 여부
3,Q019,작업점 하부 가연물의 불티 착화 우려 여부
4,Q020,금속 칸막이·벽·천장·지붕 반대면 가연물의 열전도·복사 착화 우려 여부
5,Q025,초기소화설비 비치·접근 가능 여부
6,Q026,작업 종료 후 잔불·불티 확인 여부


## 다음 단계

1. 이 노트북에서 커널을 재시작하고 모두 실행해 같은 결과가 나오는지 확인합니다.
2. 주요 위험요인에 연결된 R 기준의 필수조치와 출처를 조회하는 기능을 만듭니다.
3. 적용조건을 구조화한 뒤 조건부 질문·'해당없음'의 타당성을 검사합니다.
4. 법령 조건식 검토가 끝난 항목부터 강제경고를 연결합니다. 미검증 상태를 '경고 없음'으로 바꾸지 않습니다.

현재 작업은 CSV 읽기와 시험 계산만 수행하며 평가 결과를 파일에 저장하지 않습니다.
